# Stock Markets Analytics Zoomcamp 2026: Homework 2

This notebook currently implements Question 1: load and filter withdrawn IPOs, classify company names, calculate offered values, and compare totals by company type.

## 1. Import Libraries and Configure the Notebook

Use pandas for the HTML table loading and tabular checks.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Load or Define Question 1 Inputs

Read the recently filed IPO table from the URL specified in the homework. The file date is parsed so the live table can be restricted to the assignment's September 11, 2026 cutoff.

In [2]:
IPOSCOOP_URL = "https://www.iposcoop.com/ipos-recently-filed/"
CUTOFF_DATE = pd.Timestamp("2026-09-11")

ipo_tables = pd.read_html(IPOSCOOP_URL)
assert len(ipo_tables) == 1, f"Expected one IPO table, got {len(ipo_tables)}"

ipos_df = ipo_tables[0].copy()
required_columns = {"File Date", "Company", "Shares (millions)", "Price Low", "Price High", "Est $ Vol (millions)", "Expected To Trade"}
missing_columns = required_columns.difference(ipos_df.columns)
assert not missing_columns, f"Missing expected columns: {sorted(missing_columns)}"

ipos_df["File Date"] = pd.to_datetime(ipos_df["File Date"], errors="coerce")
print(f"Loaded table shape: {ipos_df.shape}")
print(f"File date range: {ipos_df['File Date'].min().date()} to {ipos_df['File Date'].max().date()}")
ipos_df.head()

Loaded table shape: (500, 10)
File date range: 2025-12-23 to 2026-09-24


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O


## 3. Implement Step 1, Question 1

Keep rows marked `Withdrawn` and filed strictly before September 11, 2026. This date condition reconstructs the assignment's stated as-of cutoff from the live, changing page.

In [3]:
withdrawn_mask = (
    ipos_df["Expected To Trade"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("withdrawn")
)
date_mask = ipos_df["File Date"].lt(CUTOFF_DATE)
withdrawn_df = ipos_df.loc[withdrawn_mask & date_mask].copy()

print(f"Withdrawn IPOs filed before {CUTOFF_DATE:%Y-%m-%d}: {len(withdrawn_df)}")
withdrawn_df[["File Date", "Company", "Expected To Trade"]].head(10)

Withdrawn IPOs filed before 2026-09-11: 31


,File Date,Company,Expected To Trade
38,2026-09-10,Motive Technologies (Withdrawn),Withdrawn
43,2026-09-04,Idea Tech Holding (Withdrawn),Withdrawn
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),Withdrawn
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),Withdrawn
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),Withdrawn
69,2026-08-24,pan-Africa Corp. (Withdrawn),Withdrawn
84,2026-08-18,Living Homeopathy (Withdrawn),Withdrawn
128,2026-07-31,Nuclea Energy (Withdrawn),Withdrawn
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),Withdrawn
163,2026-07-08,Cloud Data Holdings (Withdrawn),Withdrawn


## 4. Check the Filtered Results

The prompt expects 32 withdrawn IPOs at this cutoff. The live page may have changed since the assignment snapshot, so the validation reports any count discrepancy without treating it as a parsing failure.

In [5]:
EXPECTED_WITHDRAWN_COUNT = 32
actual_count = len(withdrawn_df)
current_withdrawn_count = int(withdrawn_mask.sum())
post_cutoff_count = int((withdrawn_mask & ipos_df["File Date"].ge(CUTOFF_DATE)).sum())

assert withdrawn_df["Expected To Trade"].astype("string").str.strip().str.casefold().eq("withdrawn").all()
assert withdrawn_df["File Date"].lt(CUTOFF_DATE).all()

print(f"Assignment checkpoint: {EXPECTED_WITHDRAWN_COUNT} withdrawn IPOs")
print(f"Current live source: {current_withdrawn_count} withdrawn total; {actual_count} filed before cutoff; {post_cutoff_count} on/after cutoff")
if actual_count != EXPECTED_WITHDRAWN_COUNT:
    print("Warning: the live page does not reproduce the assignment's historical 32-row checkpoint.")

withdrawn_df

Assignment checkpoint: 32 withdrawn IPOs
Current live source: 34 withdrawn total; 31 filed before cutoff; 3 on/after cutoff


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O
69,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,Santander,12.50,$10.00,$10.00,$125.00,Withdrawn,S/O
84,2026-08-18,Living Homeopathy (Withdrawn),LHI,Revere Securities,3.75,$4.00,$6.00,$19.00,Withdrawn,S/O
128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,$50.40,Withdrawn,S/O
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,US Tiger Securities,3.75,$4.00,$4.00,$15.00,Withdrawn,S/O
163,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,$15.94,Withdrawn,S/O


## 5. Classify Companies and Aggregate Withdrawn Value

Apply the company-type rules in order, parse prices and volumes, use estimated volume when shares times average price is unavailable, and group the resulting values.

In [6]:
def classify_company(company_name):
    name = "" if pd.isna(company_name) else str(company_name).casefold()
    if "technologies" in name:
        return "Technologies"
    if any(pattern in name for pattern in ("acquisition corp", "acquisition corporation", "corp")):
        return "Acquisition Corp"
    if "inc" in name or "incorporated" in name:
        return "Inc."
    if "group" in name:
        return "Group"
    if "ltd" in name or "limited" in name:
        return "Limited"
    if "holdings" in name or "holding" in name:
        return "Holdings"
    return "Other"


def parse_number(value):
    if pd.isna(value):
        return float("nan")
    text = str(value).strip().replace("$", "").replace(",", "")
    if text in {"", "-"}:
        return float("nan")
    return pd.to_numeric(text, errors="coerce")


withdrawn_df["Company Type"] = withdrawn_df["Company"].map(classify_company)
price_low = withdrawn_df["Price Low"].map(parse_number)
price_high = withdrawn_df["Price High"].map(parse_number)
withdrawn_df["Avg_price"] = (price_low + price_high) / 2
withdrawn_df["Shares (millions)"] = withdrawn_df["Shares (millions)"].map(parse_number)
withdrawn_df["Est $ Vol (millions)"] = withdrawn_df["Est $ Vol (millions)"].map(parse_number)

shares_value = withdrawn_df["Shares (millions)"] * withdrawn_df["Avg_price"]
withdrawn_df["Shares_offered_value"] = shares_value.where(
    shares_value.notna(), withdrawn_df["Est $ Vol (millions)"]
)

summary = (
    withdrawn_df.groupby("Company Type")["Shares_offered_value"]
    .sum(min_count=1)
    .sort_values(ascending=False)
    .rename("Total withdrawal value ($ millions)")
    .to_frame()
)
top_company_type = summary["Total withdrawal value ($ millions)"].idxmax()
top_value = summary.loc[top_company_type, "Total withdrawal value ($ millions)"]

print(f"Top company type: {top_company_type}")
print(f"Total withdrawal value: ${top_value:.3f} million")
summary

Top company type: Acquisition Corp
Total withdrawal value: $499.985 million


,Total withdrawal value ($ millions)
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000


# Question 2: Median Sharpe Ratio for 2025 IPOs

## 6. Load and Filter 2025 IPOs

Load the IPOScoop pricings table, keep IPOs offered before September 1, 2025, and exclude rows with a zero percent return. Check the prompt's 231-row source and 148-row filtered checkpoints before downloading prices.

In [8]:
Q2_PRICINGS_URL = "https://www.iposcoop.com/2025-pricings/"
q2_tables = pd.read_html(Q2_PRICINGS_URL)
required_q2_columns = {"Company", "Symbol", "Offer Date", "Return"}
q2_table = next(
    (table.copy() for table in q2_tables if required_q2_columns.issubset(table.columns)),
    None,
)
assert q2_table is not None, "Could not find the 2025 IPO table with Offer Date and Return columns"

q2_table["Offer Date"] = pd.to_datetime(q2_table["Offer Date"], errors="coerce")
q2_table["Return_pct"] = pd.to_numeric(
    q2_table["Return"].astype("string").str.replace("%", "", regex=False),
    errors="coerce",
)

OFFER_DATE_CUTOFF = pd.Timestamp("2025-09-01")
q2_date_mask = q2_table["Offer Date"].lt(OFFER_DATE_CUTOFF)
q2_filtered = q2_table.loc[
    q2_date_mask & q2_table["Return_pct"].ne(0)
].copy()

expected_q2_source_count = 231
expected_q2_filtered_count = 148
zero_return_before_cutoff = int((q2_date_mask & q2_table["Return_pct"].eq(0)).sum())
print(f"Source IPO rows: {len(q2_table)} (expected {expected_q2_source_count})")
print(f"Before date cutoff: {int(q2_date_mask.sum())}")
print(f"Zero-return rows before cutoff: {zero_return_before_cutoff}")
print(f"Filtered IPO rows: {len(q2_filtered)} (expected {expected_q2_filtered_count})")
assert len(q2_table) == expected_q2_source_count, f"Expected 231 IPO rows, got {len(q2_table)}"
assert q2_filtered["Offer Date"].lt(OFFER_DATE_CUTOFF).all()
assert q2_filtered["Return_pct"].ne(0).all()
if len(q2_filtered) != expected_q2_filtered_count:
    print("Warning: the live pricing table no longer matches the assignment's historical 148-row checkpoint.")
q2_filtered.head()

Source IPO rows: 231 (expected 231)
Before date cutoff: 163
Zero-return rows before cutoff: 17
Filtered IPO rows: 146 (expected 148)


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating,Return_pct
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00%,S/O,-58.0
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00%,S/O,-90.0
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75%,S/O,636.75
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75%,S/O,-90.75
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20%,S/O,1.2


## 7. Download Daily Stock Data

Download price history through September 11, 2026. Reshape Yahoo Finance's multi-level columns into one row per date and ticker, then exclude symbols with no returned closing prices.

In [9]:
import numpy as np
import yfinance as yf

q2_tickers = (
    q2_filtered["Symbol"]
    .astype("string")
    .str.strip()
    .dropna()
    .loc[lambda symbols: symbols.ne("")]
    .drop_duplicates()
    .tolist()
)

stocks_raw = yf.download(
    tickers=q2_tickers,
    start="2025-01-01",
    end="2026-09-12",
    interval="1d",
    auto_adjust=False,
    group_by="column",
    progress=False,
    threads=True,
)
assert not stocks_raw.empty, "Yahoo Finance returned no stock data"

if isinstance(stocks_raw.columns, pd.MultiIndex):
    close_levels = [
        level
        for level in range(stocks_raw.columns.nlevels)
        if "Close" in stocks_raw.columns.get_level_values(level).astype(str)
    ]
    assert len(close_levels) == 1, f"Could not identify the Close column level: {stocks_raw.columns.names}"
    ticker_level = 1 - close_levels[0]
    stocks_df = stocks_raw.stack(level=ticker_level, future_stack=True)
else:
    stocks_df = stocks_raw.copy()
    stocks_df["Ticker"] = q2_tickers[0]
    stocks_df = stocks_df.set_index("Ticker", append=True)

stocks_df.index.names = ["Date", "Ticker"]
stocks_df = stocks_df.sort_index().dropna(subset=["Close"])
available_tickers = stocks_df.index.get_level_values("Ticker").unique().tolist()
missing_tickers = sorted(set(q2_tickers).difference(available_tickers))

print(f"Requested tickers: {len(q2_tickers)}")
print(f"Tickers with Close data: {len(available_tickers)} (expected about 134)")
print(f"Tickers with no Close data: {len(missing_tickers)}")
print(f"Date range downloaded: {stocks_df.index.get_level_values('Date').min().date()} to {stocks_df.index.get_level_values('Date').max().date()}")
stocks_df.head()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CEPT"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TBH"}}}
$CEPT: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$MJID: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found

14 Failed downloads:
['CEPT', 'TBH', 'MTSR', 'AHL', 'PTNM', 'SKBL', 'EMPG', 'EPWK', 'SDM', 'MCTR', 'CAEP', 'AGH', 'MJID', 'WGRX']: pos

Requested tickers: 146
Tickers with Close data: 132 (expected about 134)
Tickers with no Close data: 14
Date range downloaded: 2025-01-02 to 2026-09-11


Price                 Adj Close         Close          High           Low          Open   Volume
Date       Ticker                                                                               
2025-01-02 ALM         1.552500      1.552500      1.552500      1.440000      1.440000  23378.0
           AVBH       21.799999     21.799999     21.799999     21.799999     21.799999      0.0
           CAPS        2.750000      2.750000      2.750000      2.500000      2.500000   2100.0
           CIIT      280.000000    280.000000    280.000000    280.000000    280.000000      0.0
           INLF    14240.000000  14240.000000  16160.000000  11232.000000  11840.000000    315.0

## 8. Calculate Growth, Volatility, and Sharpe Ratio

Compute each time-series feature independently by ticker, then describe the values available on September 11, 2026.

In [12]:
close_by_ticker = stocks_df.groupby(level="Ticker", sort=False)["Close"]
stocks_df["growth_252d"] = close_by_ticker.transform(
    lambda close: close / close.shift(252)
)
stocks_df["volatility"] = close_by_ticker.transform(
    lambda close: close.rolling(30).std() * np.sqrt(252)
)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

AS_OF_DATE = pd.Timestamp("2026-09-11")
as_of_df = stocks_df.xs(AS_OF_DATE, level="Date").copy()
sharpe_median = as_of_df["Sharpe"].median()
growth_milestone_count = int(as_of_df["growth_252d"].notna().sum())

print(f"Stocks with a 252-day growth value: {growth_milestone_count}")
print(f"Median growth: {as_of_df['growth_252d'].median():.6f}")
print(f"Mean growth: {as_of_df['growth_252d'].mean():.6f}")
print(f"Median Sharpe ratio on {AS_OF_DATE:%Y-%m-%d}: {sharpe_median:.6f}")
as_of_df[["growth_252d", "volatility", "Sharpe"]].describe()

Stocks with a 252-day growth value: 130
Median growth: 0.594534
Mean growth: 1.052117
Median Sharpe ratio on 2026-09-11: 0.049016


c:\Users\atimokhin\OneDrive - Align Technology, Inc\Documents\Private\stock-markets-analytics-zoomcamp\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Price,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.052117,27.571999,inf
std,3.049217,66.868036,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.049016
75%,1.038821,23.968537,0.133619
max,33.638270,627.698106,inf


In [14]:
# Q2 Step 6: filter to the requested trading day and describe numeric columns.
q2_step6_df = stocks_df.xs(pd.Timestamp("2026-09-11"), level="Date").copy()
q2_step6_stats = q2_step6_df.describe()

print(f"Rows on 2026-09-11: {len(q2_step6_df)}")
print(f"Median Sharpe: {q2_step6_stats.loc['50%', 'Sharpe']:.6f}")
q2_step6_stats

Rows on 2026-09-11: 132
Median Sharpe: 0.049016


c:\Users\atimokhin\OneDrive - Align Technology, Inc\Documents\Private\stock-markets-analytics-zoomcamp\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Price,Adj Close,Close,High,Low,Open,Volume,growth_252d,volatility,Sharpe
count,132.000000,132.000000,132.000000,132.000000,132.000000,1.320000e+02,130.000000,131.000000,130.000000
mean,12.774863,12.778508,13.158325,12.489526,12.810645,1.248286e+06,1.052117,27.571999,inf
std,19.149571,19.153215,19.582944,18.886522,19.231335,3.184344e+06,3.049217,66.868036,NaN
min,0.125000,0.125000,0.129000,0.115000,0.116000,0.000000e+00,0.001005,0.000000,-0.040147
25%,2.077500,2.077500,2.235750,2.019000,2.070000,2.297500e+04,0.141035,2.301309,0.012042
50%,4.565000,4.565000,4.790000,4.475000,4.505000,1.708500e+05,0.594534,7.885748,0.049016
75%,17.224999,17.224999,17.471250,16.732501,17.397501,9.976250e+05,1.038821,23.968537,0.133619
max,120.425003,120.425003,120.500000,120.425003,120.500000,2.168250e+07,33.638270,627.698106,inf


In [16]:
# Q2 observations: compare growth distribution and finite Sharpe leaders.
q2_growth = q2_step6_df["growth_252d"].dropna()
q2_comparison = q2_step6_df[["growth_252d", "Sharpe"]].replace([np.inf, -np.inf], np.nan)
q2_valid_sharpe = q2_comparison.dropna(subset=["Sharpe"])

print(f"252-day growth observations: {len(q2_growth)}")
print(f"Growth mean: {q2_growth.mean():.6f}")
print(f"Growth median: {q2_growth.median():.6f}")
print(f"Finite Sharpe observations: {len(q2_valid_sharpe)}")
print("Highest finite Sharpe ratios:")
display(q2_valid_sharpe.nlargest(5, "Sharpe").round(4))

print("Highest growth ratios:")
q2_growth_leaders = q2_growth.nlargest(5).index
q2_comparison.loc[q2_growth_leaders].round(4)

252-day growth observations: 130
Growth mean: 1.052117
Growth median: 0.594534
Finite Sharpe observations: 128
Highest finite Sharpe ratios:


Price,growth_252d,Sharpe
Ticker,,
MAGH,5.4516,4.195710e+06
HCMAU,1.0346,5.385300e+00
CEPF,1.0242,2.771000e+00
TMDE,0.7762,1.751200e+00
TLIH,0.9897,1.448900e+00


Highest growth ratios:


Price,growth_252d,Sharpe
Ticker,,
MGRT,33.6383,2.562000e-01
MAMK,6.5327,NaN
BUUU,5.9309,8.430000e-02
MAGH,5.4516,4.195710e+06
ALM,3.4809,9.430000e-02


In [11]:
adjusted_close_by_ticker = stocks_df.groupby(level="Ticker", sort=False)["Adj Close"]
adjusted_growth = adjusted_close_by_ticker.transform(
    lambda close: close / close.shift(252)
)
adjusted_volatility = adjusted_close_by_ticker.transform(
    lambda close: close.rolling(30).std() * np.sqrt(252)
)
adjusted_sharpe = (adjusted_growth - 0.05) / adjusted_volatility
adjusted_as_of = adjusted_sharpe.xs(AS_OF_DATE, level="Date")

print(f"Raw Close median Sharpe: {as_of_df['Sharpe'].median():.6f}")
print(f"Adjusted Close median Sharpe: {adjusted_as_of.median():.6f}")
print(f"Adjusted Close non-finite Sharpe values: {int((~np.isfinite(adjusted_as_of)).sum())}")

Raw Close median Sharpe: 0.049016
Adjusted Close median Sharpe: 0.050142
Adjusted Close non-finite Sharpe values: 4


In [13]:
# Conventional trailing 30-trading-day Sharpe from adjusted daily returns.
risk_free_annual = 0.05
risk_free_daily = risk_free_annual / 252

adjusted_daily_returns = stocks_df.groupby(level="Ticker", sort=False)["Adj Close"].pct_change(fill_method=None)
rolling_mean_excess = (adjusted_daily_returns - risk_free_daily).groupby(
    level="Ticker", sort=False
).transform(lambda returns: returns.rolling(30).mean())
rolling_return_std = adjusted_daily_returns.groupby(
    level="Ticker", sort=False
).transform(lambda returns: returns.rolling(30).std())
conventional_sharpe = rolling_mean_excess / rolling_return_std * np.sqrt(252)

conventional_as_of = conventional_sharpe.xs(AS_OF_DATE, level="Date")
conventional_as_of = conventional_as_of.where(np.isfinite(conventional_as_of)).dropna()
print(f"Valid conventional Sharpe values: {len(conventional_as_of)}")
print(f"Median trailing 30-day annualized Sharpe: {conventional_as_of.median():.6f}")

Valid conventional Sharpe values: 129
Median trailing 30-day annualized Sharpe: 0.290150


# Question 3: Fixed Months Holding Strategy

## Step 1: Reuse the Q2 stock-price dataset

Use the existing `stocks_df` of daily OHLCV data for the filtered IPO tickers. Verify its index, coverage, and required price column before engineering forward growth.

In [15]:
assert isinstance(stocks_df, pd.DataFrame)
assert isinstance(stocks_df.index, pd.MultiIndex)
assert stocks_df.index.names == ["Date", "Ticker"]
assert "Close" in stocks_df.columns

q3_ticker_count = stocks_df.index.get_level_values("Ticker").nunique()
q3_start_date = stocks_df.index.get_level_values("Date").min()
q3_end_date = stocks_df.index.get_level_values("Date").max()

print(f"Rows: {len(stocks_df):,}")
print(f"Tickers: {q3_ticker_count}")
print(f"Date range: {q3_start_date:%Y-%m-%d} to {q3_end_date:%Y-%m-%d}")
print(f"Columns: {', '.join(stocks_df.columns)}")
stocks_df.head()

Rows: 44,821
Tickers: 132
Date range: 2025-01-02 to 2026-09-11
Columns: Adj Close, Close, High, Low, Open, Volume, growth_252d, volatility, Sharpe


Price                 Adj Close         Close          High           Low          Open   Volume  growth_252d  \
Date       Ticker                                                                                               
2025-01-02 ALM         1.552500      1.552500      1.552500      1.440000      1.440000  23378.0          NaN   
           AVBH       21.799999     21.799999     21.799999     21.799999     21.799999      0.0          NaN   
           CAPS        2.750000      2.750000      2.750000      2.500000      2.500000   2100.0          NaN   
           CIIT      280.000000    280.000000    280.000000    280.000000    280.000000      0.0          NaN   
           INLF    14240.000000  14240.000000  16160.000000  11232.000000  11840.000000    315.0          NaN   

Price              volatility  Sharpe  
Date       Ticker                      
2025-01-02 ALM            NaN     NaN  
           AVBH           NaN     NaN  
           CAPS           NaN     NaN  
           CIIT           NaN     NaN  
           INLF           NaN     NaN

## Step 2: Engineer 1- to 12-month future growth

For each ticker and row, compare the closing price 21 to 252 ticker-specific trading observations ahead with the current closing price. Rows without a future observation at that horizon remain missing.

In [17]:
assert stocks_df.index.is_monotonic_increasing

q3_close_by_ticker = stocks_df.groupby(level="Ticker", sort=False)["Close"]
q3_future_growth_columns = []
for months in range(1, 13):
    horizon_days = months * 21
    column = f"future_growth_{months}_m"
    stocks_df[column] = q3_close_by_ticker.transform(
        lambda close, periods=horizon_days: close.shift(-periods) / close
    )
    q3_future_growth_columns.append(column)

assert len(q3_future_growth_columns) == 12
assert all(column in stocks_df.columns for column in q3_future_growth_columns)

q3_sample_ticker = stocks_df.index.get_level_values("Ticker")[0]
q3_sample_prices = stocks_df.xs(q3_sample_ticker, level="Ticker")["Close"]
q3_sample_date = q3_sample_prices.index[0]
q3_sample_actual = stocks_df.loc[(q3_sample_date, q3_sample_ticker), "future_growth_1_m"]
q3_sample_expected = q3_sample_prices.iloc[21] / q3_sample_prices.iloc[0]
assert np.isclose(q3_sample_actual, q3_sample_expected)

print(f"Created {len(q3_future_growth_columns)} future-growth columns")
print(f"Sample ticker check ({q3_sample_ticker}, 1 month): {q3_sample_actual:.6f}")
print("Non-null values by horizon:")
stocks_df[q3_future_growth_columns].notna().sum()

Created 12 future-growth columns
Sample ticker check (ALM, 1 month): 1.428985
Non-null values by horizon:


Price
future_growth_1_m     42049
future_growth_2_m     39291
future_growth_3_m     36540
future_growth_4_m     33789
future_growth_5_m     31038
future_growth_6_m     28287
future_growth_7_m     25536
future_growth_8_m     22785
future_growth_9_m     20034
future_growth_10_m    17283
future_growth_11_m    14532
future_growth_12_m    11788
dtype: int64

## Step 3: Identify each ticker's entry point

Find each ticker's earliest available trading date and the closing price on that date. This creates one IPO-entry record per ticker.

In [18]:
q3_entry_points = (
    stocks_df.reset_index()[["Date", "Ticker", "Close"]]
    .sort_values(["Ticker", "Date"])
    .groupby("Ticker", as_index=False, sort=False)
    .first()
    .rename(columns={"Date": "min_date", "Close": "entry_close"})
    .set_index("Ticker")
)

assert q3_entry_points.index.is_unique
assert len(q3_entry_points) == stocks_df.index.get_level_values("Ticker").nunique()

expected_min_dates = stocks_df.groupby(level="Ticker").apply(
    lambda ticker_data: ticker_data.index.get_level_values("Date").min(),
    include_groups=False,
)
expected_min_dates.index.name = "Ticker"
assert q3_entry_points["min_date"].sort_index().equals(expected_min_dates.sort_index())

print(f"Entry points: {len(q3_entry_points)} tickers")
q3_entry_points.head(10)

Entry points: 132 tickers


Price,min_date,entry_close
Ticker,,
AAPG,2025-01-24,17.379999
AARD,2025-02-13,14.310000
ADVB,2025-03-06,73.000000
AII,2025-05-08,16.900000
AIRO,2025-06-13,24.000000
ALM,2025-01-02,1.552500
AMBQ,2025-07-30,38.529999
ANPA,2025-07-08,3.520000
ANTA,2025-05-13,12.800000


## Step 4: Align entry points with future growth

Inner-join each ticker's entry record to the future-growth row for that same ticker and `min_date`. This gives each stock's 1- to 12-month returns starting at its first available trading day.

In [19]:
q3_growth_at_dates = (
    stocks_df.reset_index()[["Date", "Ticker", *q3_future_growth_columns]]
    .rename(columns={"Date": "min_date"})
)
q3_entry_growth = q3_entry_points.reset_index().merge(
    q3_growth_at_dates,
    on=["Ticker", "min_date"],
    how="inner",
    validate="one_to_one",
).set_index("Ticker")

assert q3_entry_growth.index.is_unique
assert len(q3_entry_growth) == len(q3_entry_points)
assert q3_entry_growth["min_date"].equals(q3_entry_points["min_date"])

q3_horizon_counts = q3_entry_growth[q3_future_growth_columns].notna().sum()
print(f"Matched entry rows: {len(q3_entry_growth)} of {len(q3_entry_points)} tickers")
print("Stocks with available future growth at each horizon:")
q3_horizon_counts

Matched entry rows: 132 of 132 tickers
Stocks with available future growth at each horizon:


Price
future_growth_1_m     132
future_growth_2_m     131
future_growth_3_m     131
future_growth_4_m     131
future_growth_5_m     131
future_growth_6_m     131
future_growth_7_m     131
future_growth_8_m     131
future_growth_9_m     131
future_growth_10_m    131
future_growth_11_m    131
future_growth_12_m    130
dtype: int64

## Step 5: Compare descriptive statistics and find the best median horizon

Use `describe()` on the 12 growth columns, then identify which holding period has the highest 50th percentile. Compare means and medians to see how outliers affect the average.

In [20]:
q3_growth_stats = q3_entry_growth[q3_future_growth_columns].describe()
q3_medians = q3_growth_stats.loc["50%"]
q3_best_column = q3_medians.idxmax()
q3_best_month = int(q3_best_column.split("_")[2])
q3_best_median_growth = q3_medians.loc[q3_best_column]

print(f"Highest median growth: month {q3_best_month}, ratio {q3_best_median_growth:.6f}")
print("Means and medians by horizon:")
q3_mean_median = q3_growth_stats.loc[["mean", "50%"]].T.rename(
    columns={"mean": "mean", "50%": "median"}
)
q3_mean_median


Highest median growth: month 1, ratio 0.935351
Means and medians by horizon:


,mean,median
Price,,
future_growth_1_m,95.751507,0.935351
future_growth_2_m,67.978625,0.892958
future_growth_3_m,51.247805,0.827160
future_growth_4_m,22.697288,0.730400
future_growth_5_m,54.242665,0.690909
future_growth_6_m,64.010844,0.700916
future_growth_7_m,59.265533,0.659973
future_growth_8_m,21.457209,0.603508
future_growth_9_m,20.006901,0.580294


In [21]:
# Question 4, Step 1: download and load the supplied indicator dataset.
import gdown
import pandas as pd

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
downloaded_file = gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "data.parquet",
    quiet=False,
)
assert downloaded_file is not None, "Google Drive download did not complete"

df = pd.read_parquet("data.parquet", engine="pyarrow")
print(f"Loaded dataset shape: {df.shape}")
print(f"Columns: {', '.join(df.columns.astype(str))}")
df.head()

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=d82edf88-e02c-4b74-b689-18199b467ca1
To: c:\Users\atimokhin\OneDrive - Align Technology, Inc\Documents\Private\stock-markets-analytics-zoomcamp\cohorts\2026\data.parquet
100%|██████████| 130M/130M [00:06<00:00, 20.3MB/s] 


Loaded dataset shape: (229932, 203)
Columns: Open, High, Low, Close_x, Volume, Dividends, Stock Splits, Ticker, Year, Month, Weekday, Date, growth_1d, growth_3d, growth_7d, growth_30d, growth_90d, growth_365d, growth_future_30d, SMA10, SMA20, growing_moving_average, high_minus_low_relative, volatility, is_positive_growth_30d_future, ticker_type, index_x, adx, adxr, apo, aroon_1, aroon_2, aroonosc, bop, cci, cmo, dx, macd, macdsignal, macdhist, macd_ext, macdsignal_ext, macdhist_ext, macd_fix, macdsignal_fix, macdhist_fix, mfi, minus_di, mom, plus_di, dm, ppo, roc, rocp, rocr, rocr100, rsi, slowk, slowd, fastk, fastd, fastk_rsi, fastd_rsi, trix, ultosc, willr, index_y, ad, adosc, obv, atr, natr, ht_dcperiod, ht_dcphase, ht_phasor_inphase, ht_phasor_quadrature, ht_sine_sine, ht_sine_leadsine, ht_trendmod, avgprice, medprice, typprice, wclprice, index, cdl2crows, cdl3blackrows, cdl3inside, cdl3linestrike, cdl3outside, cdl3starsinsouth, cdl3whitesoldiers, cdlabandonedbaby, cdladvancedblock

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,Weekday,Date,growth_1d,growth_3d,growth_7d,growth_30d,growth_90d,growth_365d,growth_future_30d,SMA10,SMA20,growing_moving_average,high_minus_low_relative,volatility,is_positive_growth_30d_future,ticker_type,index_x,adx,adxr,apo,aroon_1,aroon_2,aroonosc,bop,cci,cmo,dx,macd,macdsignal,macdhist,macd_ext,macdsignal_ext,macdhist_ext,macd_fix,macdsignal_fix,macdhist_fix,mfi,minus_di,mom,plus_di,dm,ppo,roc,rocp,rocr,rocr100,rsi,slowk,slowd,fastk,fastd,fastk_rsi,fastd_rsi,trix,ultosc,willr,index_y,ad,adosc,obv,atr,natr,ht_dcperiod,ht_dcphase,ht_phasor_inphase,ht_phasor_quadrature,ht_sine_sine,ht_sine_leadsine,ht_trendmod,avgprice,medprice,typprice,wclprice,index,cdl2crows,cdl3blackrows,cdl3inside,cdl3linestrike,cdl3outside,cdl3starsinsouth,cdl3whitesoldiers,cdlabandonedbaby,cdladvancedblock,cdlbelthold,cdlbreakaway,cdlclosingmarubozu,cdlconcealbabyswall,cdlcounterattack,cdldarkcloudcover,cdldoji,cdldojistar,cdldragonflydoji,cdlengulfing,cdleveningdojistar,cdleveningstar,cdlgapsidesidewhite,cdlgravestonedoji,cdlhammer,cdlhangingman,cdlharami,cdlharamicross,cdlhighwave,cdlhikkake,cdlhikkakemod,cdlhomingpigeon,cdlidentical3crows,cdlinneck,cdlinvertedhammer,cdlkicking,cdlkickingbylength,cdlladderbottom,cdllongleggeddoji,cdllongline,cdlmarubozu,cdlmatchinglow,cdlmathold,cdlmorningdojistar,cdlmorningstar,cdlonneck,cdlpiercing,cdlrickshawman,cdlrisefall3methods,cdlseparatinglines,cdlshootingstar,cdlshortline,cdlspinningtop,cdlstalledpattern,cdlsticksandwich,cdltakuru,cdltasukigap,cdlthrusting,cdltristar,cdlunique3river,cdlupsidegap2crows,cdlxsidegap3methods,growth_dax_1d,growth_dax_3d,growth_dax_7d,growth_dax_30d,growth_dax_90d,growth_dax_365d,growth_snp500_1d,growth_snp500_3d,growth_snp500_7d,growth_snp500_30d,growth_snp500_90d,growth_snp500_365d,growth_dji_1d,growth_dji_3d,growth_dji_7d,growth_dji_30d,growth_dji_90d,growth_dji_365d,growth_epi_1d,growth_epi_3d,growth_epi_7d,growth_epi_30d,growth_epi_90d,growth_epi_365d,Quarter,gdppot_us_yoy,gdppot_us_qoq,cpi_core_yoy,cpi_core_mom,FEDFUNDS,DGS1,DGS5,DGS10,Close_y,growth_gold_1d,growth_gold_3d,growth_gold_7d,growth_gold_30d,growth_gold_90d,growth_gold_365d,growth_wti_oil_1d,growth_wti_oil_3d,growth_wti_oil_7d,growth_wti_oil_30d,growth_wti_oil_90d,growth_wti_oil_365d,growth_brent_oil_1d,growth_brent_oil_3d,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
0,0.054277,0.062259,0.054277,0.059598,1.031789e+09,0.0,0.0,MSFT,1986,1986-03-01,3,1986-03-13,NaN,NaN,NaN,NaN,NaN,NaN,0.982144,NaN,NaN,0,0.133931,NaN,0,US,0,NaN,NaN,NaN,NaN,NaN,NaN,0.666616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3.438241e+08,NaN,1.031789e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.057602,0.058268,0.058711,0.058933,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,1.002795,1.029173,1.039264,1.108897,1.217512,1.416535,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1986-01-01,0.035188,0.008457,0.040703,0.002674,7.48,7.07,7.46,7.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.059598,0.062791,0.059598,0.061726,3.081600e+08,0.0,0.0,MSFT,1986,1986-03-01,4,1986-03-14,1.035712,NaN,NaN,NaN,NaN,NaN,0.922418,NaN,NaN,0,0.051731,NaN,0,US,1,NaN,NaN,NaN,NaN,NaN,NaN,0.666539,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,4.464654e+08,NaN,1.339949e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.060928,0.061194,0.061371,0.061460,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,1.014409,1.020976,1.054426,1.130034,1.236863,

In [22]:
# Question 4, Step 2: define the oversold RSI threshold.
RSI_THRESHOLD = 30
assert "rsi" in df.columns
assert "Date" in df.columns
assert "growth_future_30d" in df.columns
print(f"Oversold signal: rsi < {RSI_THRESHOLD}")

Oversold signal: rsi < 30


In [23]:
# Question 4, Step 3: select oversold signals within the requested date range.
Q4_START_DATE = pd.Timestamp("2000-01-01")
Q4_END_DATE = pd.Timestamp("2025-06-01")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
q4_date_mask = df["Date"].between(Q4_START_DATE, Q4_END_DATE, inclusive="both")
q4_signal_mask = df["rsi"].lt(RSI_THRESHOLD)
selected_df = df.loc[q4_date_mask & q4_signal_mask].copy()

q4_returns = selected_df["growth_future_30d"].dropna()
q4_win_rate = q4_returns.gt(1).mean()
print(f"Oversold signal rows in date range: {len(selected_df):,}")
print(f"Rows with a 30-day forward return: {len(q4_returns):,}")
print(f"Mean 30-day growth ratio: {q4_returns.mean():.6f}")
print(f"Win rate (growth_future_30d > 1): {q4_win_rate:.2%}")

Oversold signal rows in date range: 5,206
Rows with a 30-day forward return: 5,206
Mean 30-day growth ratio: 1.012640
Win rate (growth_future_30d > 1): 55.13%


In [24]:
# Question 4, Step 4: calculate cumulative net income at $1,000 per signal.
net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
net_income_thousands = net_income / 1000

print(f"Net income: ${net_income:,.2f}")
print(f"Net income ($ thousands): {net_income_thousands:,.3f}")

Net income: $65,805.59
Net income ($ thousands): 65.806


# Question 5: Improving IPO Strategy

Rather than buying every IPO, I would screen for sound fundamentals, reasonable valuation, and strong demand, then wait for early volatility to settle before entering. I would diversify positions and test the strategy with transaction costs and delisted stocks included. This may improve results, but cannot guarantee profit.